# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [2]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
466,0.69,735.0,294.0,220.5,3.5,0.25,14.250
757,0.66,759.5,318.5,220.5,3.5,0.40,16.355
680,0.86,588.0,294.0,147.0,7.0,0.40,31.955
599,0.76,661.5,416.5,122.5,7.0,0.40,40.060
459,0.74,686.0,245.0,220.5,3.5,0.25,13.890


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

> MSE kayıp fonksiyonu üzerinde eğitirdim. Amacımız bitkilerin yüksek sıcaklık farklarına ulaşmalarını engellemek, bu sebeple büyük hataları daha acımasızca cezalandıran MSE'yi tercih ederdim.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [3]:
from sklearn.preprocessing import StandardScaler

# Bağımlı ve bağımsız değişkenleri ayırma (Sütun adının Average_temperature olduğunu varsayıyoruz)
X = data.drop('Average Temperature', axis=1)
y = data['Average Temperature']

# Ölçekleyiciyi tanımlama ve tek adımda uygulama
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [4]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate
import numpy as np

# Modeli MSE (varsayılan) ile tanımlıyoruz
model_mse = SGDRegressor()

# 10 katlı çapraz doğrulama ile R2 ve Maksimum Hata testlerine sokuyoruz
cv_mse = cross_validate(model_mse, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

# Sonuçları değişkenlere atıyoruz
r2 = np.mean(cv_mse['test_r2'])
max_error_celsius = -np.min(cv_mse['test_max_error'])

print(f"MSE Modeli R2 Skoru: {r2:.4f}")
print(f"MSE Modeli Maksimum Hata: {max_error_celsius:.2f} °C")

MSE Modeli R2 Skoru: 0.8979
MSE Modeli Maksimum Hata: 9.90 °C


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [5]:
# Modeli MAE gibi davranması için manuel ayarlıyoruz
model_mae = SGDRegressor(loss='epsilon_insensitive', epsilon=0)

# Aynı test cihazına (cross_validate) tekrar sokuyoruz
cv_mae = cross_validate(model_mae, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

# Sonuçları yeni değişkenlere atıyoruz
r2_mae = np.mean(cv_mae['test_r2'])
max_error_mae = -np.min(cv_mae['test_max_error'])

print(f"MAE Modeli R2 Skoru: {r2_mae:.4f}")
print(f"MAE Modeli Maksimum Hata: {max_error_mae:.2f} °C")

MAE Modeli R2 Skoru: 0.8760
MAE Modeli Maksimum Hata: 11.22 °C


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> Bitkilerin sıcaklığa hassasiyetini göz önünde bulundurarak, far küçük de olsa daha yüksek R2 ve daha düşük max error sonucunu veren MSE ile eğitilmiş modeli kullanmayı tercih ederim.

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [6]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/didemarslan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/didemarslan/reading_the_green_line/S16D1-S-data-knn/S16D3-S-data-electrocardiograms/S16D3-S-data-threshold/S16D4-S-loss-functions/tests
plugins: dash-4.4.1, langsmith-0.12.4, typeguard-4.4.2, anyio-4.15.1
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.11s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

